# Price Elasticity & Scenario Testing
**Marketing Analytics Module 3** — Scan\*Pro log-log OLS · Price elasticity · What-if scenario simulation

Models demand sensitivity to price changes for each of 44 SKUs using a Scan\*Pro log-log regression.
Produces elasticity estimates, demand/revenue curves, and ±5/10/20/30% price scenario simulations.

$$\log(\text{sales}) = \beta_0 + \underbrace{\beta_1}_{\varepsilon} \cdot \log(\text{price}) + \beta_2 \cdot \text{feat\_main\_page} + \text{controls} + \varepsilon_t$$


In [23]:
# Load all libraries needed for modelling, charts, and interactive widgets
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import statsmodels.api as sm
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

pio.templates.default = "plotly_white"

# ── Single blue accent palette (professional/corporate) ───────────────────────
# Inspired by Tufte's data-ink ratio principle — colour only where it carries meaning
BLUE     = "#2563EB"          # Primary accent — key data points, best model
BLUE_LT  = "rgba(37,99,235,0.10)"  # CI band fill — extremely subtle
SLATE    = "#0f172a"          # Near-black — actual/observed data
GREY_LG  = "#cbd5e1"          # Light grey — historical / secondary traces
GREY_MD  = "#94a3b8"          # Mid grey   — gridlines, annotations
RED      = "#DC2626"          # Negative Δ revenue only
GREEN    = "#16a34a"          # Positive Δ revenue only

# ── KPI card helpers — identical style to Module 1 & 2 ───────────────────────
def metric_card(label, value, sub=""):
    return f'''<div style="flex:1;min-width:160px;background:#f8fafc;
    border:1px solid #e2e8f0;border-radius:12px;padding:1.1rem 1.4rem;">
    <div style="font-size:0.72rem;color:#64748b;text-transform:uppercase;
    letter-spacing:0.05em;font-weight:500;">{label}</div>
    <div style="font-size:1.45rem;font-weight:700;color:#0f172a;
    margin-top:0.2rem;font-family:monospace;">{value}</div>
    <div style="font-size:0.78rem;color:#64748b;margin-top:0.15rem;">{sub}</div>
    </div>'''

def metric_row(cards):
    inner = "".join(cards)
    return HTML(f'<div style="display:flex;gap:1rem;margin:1rem 0;'
                f'flex-wrap:wrap;">{inner}</div>')




In [25]:
# Load datasets and build SKU name lookup from raw data
proc = pd.read_csv("data_processed.csv")
proc["week"] = pd.to_datetime(proc["week"])
proc = proc.sort_values(["sku", "week"]).reset_index(drop=True)

# Build SKU metadata from raw data (names, colours, categories)
raw = pd.read_csv("data_raw.csv")
raw["week"] = pd.to_datetime(raw["week"])
raw["feat_main_page"] = raw["feat_main_page"].astype(str).str.lower().eq("true").astype(int)

sku_meta = raw.groupby("sku").agg(
    functionality=("functionality", "first"),
    color=("color", "first"),
    vendor=("vendor", "first"),
    avg_price=("price", "mean"),
    avg_sales=("weekly_sales", "mean"),
    promo_rate=("feat_main_page", "mean"),
).reset_index()

# Clean category — strip numeric prefix (e.g. "06.Mobile phone accessories" → "Mobile phone accessories")
sku_meta["category"] = sku_meta["functionality"].str.replace(
    r"^\d+\.", "", regex=True).str.strip()

# Human-readable display name: "Mobile phone accessories (Black)"
sku_meta["display_name"] = sku_meta.apply(
    lambda r: f"{r['category']} ({r['color'].title()})", axis=1)

# Full dropdown label: "SKU 1 — Mobile phone accessories (Black)"
sku_meta["label"] = sku_meta.apply(
    lambda r: f"SKU {r['sku']} — {r['display_name']}", axis=1)

def sku_label(sku_id):
    row = sku_meta[sku_meta["sku"] == sku_id]
    return row["label"].iloc[0] if len(row) > 0 else f"SKU {sku_id}"

def sku_category(sku_id):
    row = sku_meta[sku_meta["sku"] == sku_id]
    return row["category"].iloc[0] if len(row) > 0 else "Unknown"

MONTH_COLS = [f"month_{i}" for i in range(2, 13)]
SKUS       = sorted(proc["sku"].unique())
SCENARIOS  = [-0.30, -0.20, -0.10, -0.05, 0.05, 0.10, 0.20, 0.30]

print(f"Loaded {proc['sku'].nunique()} SKUs · {proc['week'].nunique()} weeks")
print(f"   Date range: {proc['week'].min().date()} → {proc['week'].max().date()}")
print(f"\nSample SKU names:")
for s in [1, 8, 10, 25, 43]:
    print(f"   {sku_label(s)}")


Loaded 44 SKUs · 98 weeks
   Date range: 2016-11-14 → 2018-09-24

Sample SKU names:
   SKU 1 — Mobile phone accessories (Black)
   SKU 8 — Bluetooth speakers (Black)
   SKU 10 — VR headset (White)
   SKU 25 — Selfie sticks (Green)
   SKU 43 — Fitness trackers (Gold)


In [26]:
# ── Scan*Pro log-log OLS — fitted per SKU ────────────────────────────────────
# β₁ = price elasticity directly from the regression coefficient

def fit_scanpro(sku_id):
    """Fit Scan*Pro for one SKU. Returns result dict or None."""
    df = proc[proc["sku"] == sku_id].sort_values("week").copy()
    df = df[df["weekly_sales"] > 0]
    if len(df) < 15 or df["feat_main_page"].nunique() < 2:
        return None

    y = np.log(df["weekly_sales"].values)
    X = pd.DataFrame(index=df.index)
    X["log_price"]      = np.log(df["price"].clip(lower=0.01))
    X["feat_main_page"] = df["feat_main_page"]
    X["trend"]          = df["trend"]
    for m in MONTH_COLS:
        if m in df.columns:
            X[m] = df[m]
    X = sm.add_constant(X, has_constant="add")

    try:
        model = sm.OLS(y, X).fit()
    except Exception:
        return None

    elast      = float(model.params.get("log_price", np.nan))
    elast_pval = float(model.pvalues.get("log_price", np.nan))
    if np.isnan(elast):
        return None

    P0 = float(df["price"].mean())
    Q0 = float(df["weekly_sales"].mean())
    R0 = P0 * Q0

    # Continuous price curve for demand/revenue charts
    price_range   = np.linspace(P0 * 0.20, P0 * 3.0, 400)
    demand_curve  = np.clip(Q0 * (price_range / P0) ** elast, 0, None)
    revenue_curve = price_range * demand_curve
    opt_idx       = int(np.argmax(revenue_curve))

    # What-if scenario table
    rows = []
    for pct in SCENARIOS:
        P1 = P0 * (1 + pct)
        Q1 = max(Q0 * ((P1 / P0) ** elast), 0)
        R1 = P1 * Q1
        rows.append({
            "Scenario"        : f"{'↑' if pct>0 else '↓'} {abs(int(pct*100))}%",
            "pct"             : pct,
            "New Price (£)"   : round(P1, 2),
            "Δ Demand (units)": round(Q1 - Q0, 1),
            "Δ Demand (%)"    : round((Q1-Q0)/Q0*100, 1) if Q0 > 0 else 0,
            "New Revenue (£)" : round(R1, 2),
            "Δ Revenue (£)"   : round(R1-R0, 2),
            "Δ Revenue (%)"   : round((R1-R0)/R0*100, 1) if R0 > 0 else 0,
        })

    return {
        "sku"            : int(sku_id),
        "name"           : sku_label(int(sku_id)),
        "category"       : sku_category(int(sku_id)),
        "elasticity"     : round(elast, 4),
        "elast_pval"     : round(elast_pval, 4),
        "elast_sig"      : bool(elast_pval < 0.05),
        "r2"             : round(float(model.rsquared), 4),
        "avg_price"      : round(P0, 2),
        "avg_sales"      : round(Q0, 1),
        "avg_revenue"    : round(R0, 2),
        "price_range"    : price_range,
        "demand_curve"   : demand_curve,
        "revenue_curve"  : revenue_curve,
        "optimal_price"  : round(float(price_range[opt_idx]), 2),
        "optimal_revenue": round(float(revenue_curve[opt_idx]), 2),
        "scenario_df"    : pd.DataFrame(rows),
        "weeks"          : df["week"].values,
        "actual"         : df["weekly_sales"].values,
    }

# Run on all SKUs
print(" Fitting Scan*Pro for all 44 SKUs...")
results = {}
for sku_id in SKUS:
    r = fit_scanpro(sku_id)
    if r:
        results[sku_id] = r

# Summary DataFrame
results_df = pd.DataFrame([{
    "sku"          : r["sku"],
    "name"         : r["name"],
    "category"     : r["category"],
    "elasticity"   : r["elasticity"],
    "elast_pval"   : r["elast_pval"],
    "elast_sig"    : r["elast_sig"],
    "r2"           : r["r2"],
    "avg_price"    : r["avg_price"],
    "avg_sales"    : r["avg_sales"],
    "avg_revenue"  : r["avg_revenue"],
    "optimal_price": r["optimal_price"],
} for r in results.values()]).sort_values("elasticity").reset_index(drop=True)

print(f" Fitted {len(results_df)} SKUs")


 Fitting Scan*Pro for all 44 SKUs...
 Fitted 44 SKUs


In [27]:
# ── Portfolio overview KPIs ───────────────────────────────────────────────────
n_elastic   = int((results_df["elasticity"] < -1).sum())
n_inelastic = int(((results_df["elasticity"] >= -1) & (results_df["elasticity"] < 0)).sum())
n_sig       = int(results_df["elast_sig"].sum())
avg_elast   = round(results_df["elasticity"].mean(), 3)
most_sens   = results_df.loc[results_df["elasticity"].idxmin()]
least_sens  = results_df.loc[results_df["elasticity"].idxmax()]

display(metric_row([
    metric_card("SKUs modelled",        f"{len(results_df)}",
                "with sufficient data"),
    metric_card("Elastic  |ε| > 1",    f"{n_elastic} / {len(results_df)}",
                "price-sensitive demand"),
    metric_card("Inelastic  |ε| < 1",  f"{n_inelastic} / {len(results_df)}",
                "price-insensitive demand"),
    metric_card("Significant  p<0.05", f"{n_sig} / {len(results_df)}",
                "statistically reliable ε"),
    metric_card("Avg elasticity",       f"{avg_elast}",
                "across all SKUs"),
    metric_card("Avg model R²",         f"{results_df['r2'].mean():.3f}",
                "Scan*Pro goodness of fit"),
]))

display(metric_row([
    metric_card("Most price-sensitive",
                f"SKU {int(most_sens['sku'])}",
                f"{most_sens['name'].split('—')[-1].strip()}  |  ε = {most_sens['elasticity']:.3f}"),
    metric_card("Least price-sensitive",
                f"SKU {int(least_sens['sku'])}",
                f"{least_sens['name'].split('—')[-1].strip()}  |  ε = {least_sens['elasticity']:.3f}"),
]))


## Portfolio overview

In [28]:
# ── Elasticity ranking — all SKUs ────────────────────────────────────────────
# Horizontal bar chart ordered by elasticity.
# Blue = significant (p<0.05) · Grey = not significant · dashed line = unit elastic

df_plot = results_df.copy()
df_plot["bar_color"] = df_plot.apply(
    lambda r: BLUE if r["elast_sig"] else GREY_MD, axis=1)
df_plot["label"] = df_plot.apply(
    lambda r: f"SKU {int(r['sku'])} — {r['name'].split('—')[-1].strip()}"
              + (" ★" if r["elast_sig"] else ""), axis=1)

fig = go.Figure(go.Bar(
    y=df_plot["label"],
    x=df_plot["elasticity"],
    orientation="h",
    marker_color=df_plot["bar_color"],
    text=df_plot["elasticity"].apply(lambda v: f"{v:.2f}"),
    textposition="outside",
    textfont=dict(size=9, color=GREY_MD),
    customdata=df_plot[["elast_pval", "avg_price", "avg_sales", "r2",
                         "elast_sig"]].values,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Elasticity: <b>%{x:.4f}</b><br>"
        "p-value: %{customdata[0]:.4f}"
        "  ·  Significant: %{customdata[4]}<br>"
        "Avg price: £%{customdata[1]:.2f}"
        "  ·  Avg sales: %{customdata[2]:.0f} units/wk<br>"
        "R²: %{customdata[3]:.3f}<extra></extra>"
    ),
))

fig.add_vline(x=-1, line_dash="dash", line_color=GREY_MD, line_width=1.2,
              annotation_text="Unit elastic  ε = −1",
              annotation_font=dict(color=GREY_MD, size=10),
              annotation_position="top right")
fig.add_vline(x=0,  line_dash="dot",  line_color=GREY_MD, line_width=0.8)

fig.update_layout(
    title=dict(text="Price Elasticity per SKU  (Scan*Pro log-log OLS)",
               font=dict(size=14, color=SLATE)),
    height=950,
    margin=dict(l=10, r=80, t=50, b=40),
    xaxis=dict(title="Price Elasticity (ε)", zeroline=True,
               zerolinecolor=GREY_LG, zerolinewidth=1,
               tickfont=dict(color=GREY_MD)),
    yaxis=dict(tickfont=dict(size=10, color=SLATE)),
    showlegend=False,
    plot_bgcolor="white",
)

# Manual legend annotation
fig.add_annotation(
    x=0.99, y=0.01, xref="paper", yref="paper",
    text="■ Blue = significant (p<0.05)  ·  ■ Grey = not significant  ·  ★ = significant",
    showarrow=False, font=dict(size=9, color=GREY_MD),
    bgcolor="white", align="right",
)

fig.show()
print(f"\n{n_elastic} elastic SKUs (|ε|>1)  ·  "
      f"{n_inelastic} inelastic  ·  "
      f"{n_sig} statistically significant")



32 elastic SKUs (|ε|>1)  ·  12 inelastic  ·  40 statistically significant


In [29]:
# ── Revenue impact heatmap — all SKUs × all 8 scenarios ──────────────────────
# Green = revenue gain · Red = revenue loss · white = near-zero change

scenario_labels = [f"{'↑' if p>0 else '↓'}{abs(int(p*100))}%" for p in SCENARIOS]
rows_sorted = results_df.sort_values("elasticity").itertuples()

sku_labels, z_matrix, text_matrix = [], [], []

for row in results_df.sort_values("elasticity").itertuples():
    r     = results[row.sku]
    P0, Q0, R0 = r["avg_price"], r["avg_sales"], r["avg_revenue"]
    zrow, trow = [], []
    for pct in SCENARIOS:
        P1   = P0 * (1 + pct)
        Q1   = max(Q0 * ((P1 / P0) ** r["elasticity"]), 0)
        dpct = (P1*Q1 - R0) / R0 * 100 if R0 > 0 else 0
        zrow.append(round(dpct, 1))
        trow.append(f"{dpct:+.1f}%")
    sku_labels.append(r["name"].replace("SKU ", ""))  # shorter label
    z_matrix.append(zrow)
    text_matrix.append(trow)

fig = go.Figure(go.Heatmap(
    z=z_matrix,
    x=scenario_labels,
    y=sku_labels,
    text=text_matrix,
    texttemplate="%{text}",
    textfont=dict(size=8, color="#334155"),
    colorscale=[
        [0.00, "#DC2626"],
        [0.40, "#FCA5A5"],
        [0.50, "#F8FAFC"],
        [0.60, "#86EFAC"],
        [1.00, "#16a34a"],
    ],
    zmid=0,
    colorbar=dict(title="ΔRevenue (%)", ticksuffix="%", len=0.7,
                  tickfont=dict(size=10)),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Scenario: %{x}<br>"
        "Revenue change: <b>%{z:+.1f}%</b><extra></extra>"
    ),
))

fig.update_layout(
    title=dict(text="Revenue Impact Heatmap  —  All SKUs × All Price Scenarios",
               font=dict(size=14, color=SLATE)),
    height=1000,
    margin=dict(l=10, r=110, t=50, b=60),
    xaxis=dict(title="Price Change Scenario", side="top",
               tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=9), autorange="reversed"),
    paper_bgcolor="white",
)

fig.show()
print("Tip: SKUs at the top are most elastic — they react strongly to price changes.")
print("     SKUs showing green at '+10%' are inelastic — they can absorb price rises.")


Tip: SKUs at the top are most elastic — they react strongly to price changes.
     SKUs showing green at '+10%' are inelastic — they can absorb price rises.


## Interactive SKU deep-dive

Select any SKU to see its demand curve, revenue curve, optimal price, and what-if scenario table.


In [30]:
# ── Interactive SKU drill-down ────────────────────────────────────────────────

sku_dd = widgets.Dropdown(
    options=[(r["name"], sku_id) for sku_id, r in
             sorted(results.items(), key=lambda x: x[0])],
    value=list(results.keys())[0],
    layout=widgets.Layout(width="420px"),
)
out = widgets.Output()

def _render(change=None):
    out.clear_output(wait=True)
    sku_id = sku_dd.value
    r      = results[sku_id]
    ε      = r["elasticity"]
    P0, Q0, R0 = r["avg_price"], r["avg_sales"], r["avg_revenue"]

    with out:
        # ── SKU header ────────────────────────────────────────────────────
        display(HTML(
            f'<div style="margin:.5rem 0 .2rem;">'
            f'<div style="font-size:1.05rem;font-weight:700;color:{SLATE};">'
            f'{r["name"]}</div>'
            f'<div style="font-size:.82rem;color:{GREY_MD};margin-top:.2rem;">'
            f'Category: {r["category"]} &nbsp;·&nbsp; '
            f'Avg price: £{P0:.2f} &nbsp;·&nbsp; '
            f'Avg sales: {Q0:.0f} units/wk &nbsp;·&nbsp; '
            f'Avg revenue: £{R0:,.0f}/wk'
            f'</div></div>'
        ))

        # ── Elasticity interpretation ─────────────────────────────────────
        if ε < -1:
            interp = (f"Elastic  |ε| = {abs(ε):.2f} > 1 — "
                      f"Demand is price-sensitive. "
                      f"A 10% price rise → {abs(ε)*10:.0f}% demand fall. "
                      f"Lowering price is likely to grow revenue.")
        elif ε < 0:
            interp = (f"Inelastic  |ε| = {abs(ε):.2f} < 1 — "
                      f"Demand is price-insensitive. "
                      f"A 10% price rise → {abs(ε)*10:.0f}% demand fall. "
                      f"Raising price is likely to grow revenue.")
        else:
            interp = (f"Positive elasticity  ε = {ε:.2f} — "
                      f"Unusual result. "
                      f"Check model significance (p = {r['elast_pval']:.4f}).")

        display(HTML(
            f'<div style="background:#f8fafc;border-left:3px solid {BLUE};'
            f'padding:.6rem 1rem;margin:.4rem 0 .8rem;border-radius:0 6px 6px 0;'
            f'font-size:.84rem;color:#334155;">{interp}</div>'
        ))

        # ── KPI cards ─────────────────────────────────────────────────────
        sig_str = "★ significant" if r["elast_sig"] else "not significant"
        display(metric_row([
            metric_card("Price elasticity", f"{ε:.4f}",
                        f"p = {r['elast_pval']:.4f}  ·  {sig_str}"),
            metric_card("Model R²",         f"{r['r2']:.4f}",
                        "Scan*Pro goodness of fit"),
            metric_card("Avg price",        f"£{P0:.2f}",
                        f"Avg sales: {Q0:.0f} units/wk"),
            metric_card("Avg revenue",      f"£{R0:,.0f}",
                        "per week at current price"),
            metric_card("Revenue-max price",f"£{r['optimal_price']:.2f}",
                        f"Peak £{r['optimal_revenue']:,.0f}/wk"),
        ]))

        # ── Demand + Revenue curves ───────────────────────────────────────
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=[
                f"Demand Curve  (ε = {ε:.3f})",
                f"Revenue Curve  —  Optimal: £{r['optimal_price']:.2f}",
            ],
            horizontal_spacing=0.10,
        )

        # Demand
        fig.add_trace(go.Scatter(
            x=r["price_range"], y=r["demand_curve"],
            mode="lines", line=dict(color=BLUE, width=2),
            name="Demand", showlegend=False,
            hovertemplate="Price: £%{x:.2f}<br>Demand: %{y:.0f} units<extra></extra>",
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=[P0], y=[Q0], mode="markers",
            marker=dict(size=11, color=SLATE,
                        line=dict(color="white", width=2)),
            name="Current price", showlegend=False,
            hovertemplate=f"Current: £{P0:.2f} → {Q0:.0f} units<extra></extra>",
        ), row=1, col=1)

        # Revenue
        fig.add_trace(go.Scatter(
            x=r["price_range"], y=r["revenue_curve"],
            mode="lines", line=dict(color=GREY_LG, width=2),
            name="Revenue curve", showlegend=False,
            hovertemplate="Price: £%{x:.2f}<br>Revenue: £%{y:,.0f}<extra></extra>",
        ), row=1, col=2)
        fig.add_trace(go.Scatter(
            x=[P0], y=[R0], mode="markers",
            marker=dict(size=11, color=SLATE,
                        line=dict(color="white", width=2)),
            name="Current", showlegend=False,
            hovertemplate=f"Current: £{P0:.2f} → £{R0:,.0f}<extra></extra>",
        ), row=1, col=2)
        fig.add_trace(go.Scatter(
            x=[r["optimal_price"]], y=[r["optimal_revenue"]],
            mode="markers",
            marker=dict(size=14, color=BLUE,
                        symbol="star", line=dict(color="white", width=1.5)),
            name=f"Optimal £{r['optimal_price']:.2f}", showlegend=False,
            hovertemplate=(f"Optimal: £{r['optimal_price']:.2f}"
                           f" → £{r['optimal_revenue']:,.0f}/wk<extra></extra>"),
        ), row=1, col=2)

        for col, ytitle in [(1, "Weekly demand (units)"),
                            (2, "Weekly revenue (£)")]:
            fig.update_xaxes(title_text="Price (£)", row=1, col=col,
                             tickfont=dict(color=GREY_MD))
            fig.update_yaxes(title_text=ytitle, row=1, col=col,
                             tickfont=dict(color=GREY_MD))

        fig.update_layout(
            height=360,
            margin=dict(t=50, b=40, l=60, r=40),
            plot_bgcolor="white",
            paper_bgcolor="white",
        )
        fig.show()

        # ── Scenario waterfall ────────────────────────────────────────────
        sc = r["scenario_df"]
        bar_colors = [RED if v < 0 else GREEN for v in sc["Δ Revenue (£)"]]

        fig2 = go.Figure(go.Bar(
            x=sc["Scenario"],
            y=sc["Δ Revenue (£)"],
            marker_color=bar_colors,
            marker_line_width=0,
            text=[f"£{v:+,.0f}" for v in sc["Δ Revenue (£)"]],
            textposition="outside",
            textfont=dict(size=10, color=GREY_MD),
            customdata=sc[["New Price (£)", "Δ Demand (%)",
                            "New Revenue (£)", "Δ Revenue (%)"]].values,
            hovertemplate=(
                "<b>%{x}</b><br>"
                "New price: £%{customdata[0]:.2f}<br>"
                "Δ Demand: %{customdata[1]:+.1f}%<br>"
                "New revenue: £%{customdata[2]:,.0f}<br>"
                "Δ Revenue: %{customdata[3]:+.1f}%<extra></extra>"
            ),
        ))
        fig2.add_hline(y=0, line_color=GREY_LG, line_width=1)
        fig2.update_layout(
            title=dict(
                text=f"Revenue Impact by Price Scenario  (ε = {ε:.3f})",
                font=dict(size=13, color=SLATE),
            ),
            xaxis=dict(title="Price change scenario",
                       tickfont=dict(color=GREY_MD)),
            yaxis=dict(title="Δ Revenue vs baseline (£/week)",
                       tickfont=dict(color=GREY_MD)),
            height=340,
            margin=dict(t=50, b=40, l=60, r=40),
            plot_bgcolor="white",
            paper_bgcolor="white",
        )
        fig2.show()

        # ── Scenario table ────────────────────────────────────────────────
        display(HTML('<div style="font-size:.88rem;font-weight:600;'
                     f'color:{SLATE};margin:.8rem 0 .4rem;">What-if scenario table</div>'))

        def _fmt_delta(val, suffix=""):
            color = RED if val < 0 else GREEN
            sign  = "+" if val >= 0 else ""
            return (f'<span style="color:{color};font-weight:600;">'
                    f'{sign}{val}{suffix}</span>')

        rows_html = ""
        for _, row in sc.iterrows():
            bg = "#fef9f9" if row["pct"] > 0 else "#f9fef9"
            rows_html += (
                f'<tr style="background:{bg}">'
                f'<td style="font-weight:600">{row["Scenario"]}</td>'
                f'<td>£{row["New Price (£)"]:.2f}</td>'
                f'<td>{_fmt_delta(row["Δ Demand (units)"])}</td>'
                f'<td>{_fmt_delta(row["Δ Demand (%)"], "%")}</td>'
                f'<td>£{row["New Revenue (£)"]:,.0f}</td>'
                f'<td>{_fmt_delta(row["Δ Revenue (£)"], "")}</td>'
                f'<td>{_fmt_delta(row["Δ Revenue (%)"], "%")}</td>'
                f'</tr>'
            )

        th = ("background:#0f172a;color:white;padding:.6rem .9rem;"
              "font-size:.75rem;text-transform:uppercase;"
              "letter-spacing:.04em;font-weight:500;text-align:center;")
        td_style = ("font-size:.84rem;padding:.55rem .9rem;"
                    "text-align:center;border-bottom:1px solid #f1f5f9;")

        display(HTML(
            f'<div style="overflow-x:auto;margin-top:.5rem">'
            f'<table style="width:100%;border-collapse:collapse;">'
            f'<thead><tr>'
            f'<th style="{th}">Scenario</th>'
            f'<th style="{th}">New Price</th>'
            f'<th style="{th}">Δ Demand (units)</th>'
            f'<th style="{th}">Δ Demand (%)</th>'
            f'<th style="{th}">New Revenue</th>'
            f'<th style="{th}">Δ Revenue (£)</th>'
            f'<th style="{th}">Δ Revenue (%)</th>'
            f'</tr></thead>'
            f'<tbody style="{td_style}">{rows_html}</tbody>'
            f'</table></div>'
        ))

        # ── Model stats footnote ──────────────────────────────────────────
        display(HTML(
            f'<div style="font-size:.76rem;color:{GREY_MD};margin-top:.8rem;">'
            f'Scan*Pro model — β₁ (price elasticity) = {ε:.4f}  ·  '
            f'p-value = {r["elast_pval"]:.4f}  ·  '
            f'R² = {r["r2"]:.4f}  ·  '
            f'{"★ Statistically significant at p<0.05" if r["elast_sig"] else "Not significant at p<0.05"}'
            f'</div>'
        ))

sku_dd.observe(_render, names="value")
display(widgets.HBox([
    widgets.Label("Select SKU:", layout=widgets.Layout(
        margin="6px 8px 0 0", font_weight="bold")),
    sku_dd,
]))
display(out)
_render()


Output()

## All-SKU price strategy summary

In [31]:
# ── All-SKU strategy table ───────────────────────────────────────────────────
# Shows ±10% revenue impact and recommended strategy per SKU

summary_rows = []
for _, row in results_df.iterrows():
    r  = results[row["sku"]]
    ε  = r["elasticity"]
    P0, Q0, R0 = r["avg_price"], r["avg_sales"], r["avg_revenue"]
    Pup, Pdn   = P0*1.10, P0*0.90
    Rup = Pup * max(Q0*((Pup/P0)**ε), 0)
    Rdn = Pdn * max(Q0*((Pdn/P0)**ε), 0)
    dUp = round((Rup-R0)/R0*100, 1) if R0>0 else 0
    dDn = round((Rdn-R0)/R0*100, 1) if R0>0 else 0
    strategy = ("Consider lower price"  if ε < -1
                else "Consider higher price" if ε < 0
                else "Review data / model")
    summary_rows.append({
        "SKU"              : f"SKU {int(row['sku'])}",
        "Product"          : r["name"].split("—")[-1].strip(),
        "Category"         : r["category"],
        "Elasticity (ε)"   : row["elasticity"],
        "Sig."             : "★" if row["elast_sig"] else "",
        "Avg Price (£)"    : row["avg_price"],
        "ΔRev +10% (%)"    : dUp,
        "ΔRev −10% (%)"    : dDn,
        "Optimal Price (£)": row["optimal_price"],
        "Strategy"         : strategy,
    })

summary_df = pd.DataFrame(summary_rows)

def _c(val):
    if isinstance(val, float):
        return f"color:{RED};font-weight:600" if val < 0                else f"color:{GREEN};font-weight:600"
    return ""

styled = (
    summary_df.style
    .applymap(_c, subset=["ΔRev +10% (%)", "ΔRev −10% (%)"])
    .applymap(lambda v: f"color:{BLUE};font-weight:700"
              if isinstance(v, float) and v < -1 else
              (f"color:{GREY_MD}" if isinstance(v, float) else ""),
              subset=["Elasticity (ε)"])
    .format({
        "Elasticity (ε)"   : "{:.4f}",
        "Avg Price (£)"    : "£{:.2f}",
        "ΔRev +10% (%)"    : "{:+.1f}%",
        "ΔRev −10% (%)"    : "{:+.1f}%",
        "Optimal Price (£)": "£{:.2f}",
    })
    .set_properties(**{
        "text-align" : "center",
        "font-size"  : "0.84rem",
        "padding"    : "6px 10px",
    })
    .set_table_styles([{
        "selector": "th",
        "props": [
            ("background-color", "#0f172a"),
            ("color", "white"),
            ("font-size", "0.75rem"),
            ("padding", "8px 10px"),
            ("text-align", "center"),
            ("text-transform", "uppercase"),
            ("letter-spacing", "0.04em"),
        ],
    }])
)

display(styled)
print(f"\n{(summary_df['ΔRev +10% (%)']<0).sum()} SKUs lose revenue at +10% price (elastic)")
print(f"{(summary_df['ΔRev +10% (%)']>0).sum()} SKUs gain revenue at +10% price (inelastic)")


,SKU,Product,Category,Elasticity (ε),Sig.,Avg Price (£),ΔRev +10% (%),ΔRev −10% (%),Optimal Price (£),Strategy
0,SKU 9,Fitness trackers (Black),Fitness trackers,-4.0713,★,£162.19,-25.4%,+38.2%,£32.44,Consider lower price
1,SKU 19,Streaming sticks (Black),Streaming sticks,-4.0068,★,£93.67,-24.9%,+37.2%,£18.73,Consider lower price
2,SKU 33,Selfie sticks (Green),Selfie sticks,-3.5109,★,£9.17,-21.3%,+30.3%,£1.83,Consider lower price
3,SKU 30,Mobile phone accessories (Grey),Mobile phone accessories,-2.9269,★,£19.02,-16.8%,+22.5%,£3.80,Consider lower price
4,SKU 11,Streaming sticks (Black),Streaming sticks,-2.9238,★,£28.72,-16.8%,+22.3%,£5.74,Consider lower price
5,SKU 25,Selfie sticks (Green),Selfie sticks,-2.8389,★,£8.42,-16.1%,+21.3%,£1.68,Consider lower price
6,SKU 34,Portable smartphone chargers (Grey),Portable smartphone chargers,-2.5384,★,£19.83,-13.6%,+17.6%,£3.97,Consider lower price
7,SKU 15,Mobile phone accessories (Red),Mobile phone accessories,-2.5265,★,£17.32,-13.5%,+17.5%,£3.46,Consider lower price
8,SKU 7,Selfie sticks (Blue),Selfie sticks,-2.4623,★,£11.32,-13.0%,+16.6%,£2.26,Consider lower price
9,SKU 24,Flash drives (None),Flash drives,-2.3642,★,£21.26,-12.3%,+15.4%,£4.25,Consider lower price



35 SKUs lose revenue at +10% price (elastic)
8 SKUs gain revenue at +10% price (inelastic)


## Methodology

**Data:** 44 SKUs × 98 weeks from `data_processed.csv` (lagged prices, trend, month dummies, one-hot categoricals).
SKU names extracted from `data_raw.csv` (functionality, colour, vendor).

**Model:** Scan\*Pro log-log OLS fitted per SKU:

$$\log(\text{sales}_t) = \beta_0 + \beta_1 \cdot \log(\text{price}_t) + \beta_2 \cdot \text{feat\_main\_page}_t + \beta_3 \cdot \text{trend}_t + \sum_{m} \gamma_m \cdot \text{month}_{m,t} + \varepsilon_t$$

The price coefficient **β₁ is the price elasticity directly** — no further transformation needed.

**Scenario simulation:** Power-law demand function derived from elasticity:
$$Q_{\text{new}} = Q_0 \times \left(\frac{P_{\text{new}}}{P_0}\right)^{\varepsilon}, \quad R_{\text{new}} = P_{\text{new}} \times Q_{\text{new}}$$

**Significance:** p-value < 0.05 on β₁ (★ in charts and tables).

**Limitations:**
- Constant elasticity assumption (log-log form) — may not hold at extreme prices
- No margin/cost floor — optimal price is purely revenue-maximising
- SKUs with limited price variation may have unreliable estimates

**Reference:** Cohen, M. C., Gras, P.E., Pentecoste, A., & Zhang, R. (2022). *Demand Prediction in Retail.* Springer.
